# Notebook 05. Đánh giá Chỉ số Digital Burnout (DBI)

Notebook này thực hiện xây dựng bộ dữ liệu đánh giá **Digital Burnout Indicator (DBI)** trên dữ liệu khảo sát sinh viên Việt Nam.

Quy trình bao gồm:

1. Tải dữ liệu khảo sát đã được tiền xử lý.
2. Tải danh sách đặc trưng đã được xác thực.
3. Tải bảng ánh xạ giữa đặc trưng và Candidate Indicators.
4. Chuẩn hóa các chỉ số thành phần.
5. Tính điểm cho từng nhóm chỉ số DBI.
6. Xuất bộ dữ liệu phục vụ xây dựng DBI Framework.

Lưu ý: Notebook này **chỉ tính điểm cho từng nhóm chỉ số (DBI Dimensions)**, chưa tính điểm DBI tổng hợp. Trọng số của từng nhóm chỉ số sẽ được xác định trong giai đoạn xây dựng **DBI Framework**.

# 0. Set Up

Thiết lập môi trường thực thi cho Notebook 04 - Feature Validation.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

print("Đã import thư viện.")

Đã import thư viện.


In [4]:
# Thiết lập đường dẫn dữ liệu

survey_path = Path(
    "../../data/processed/vietnam_dataset/vn_digital_burnout_cleaned.csv"
)

validated_feature_path = Path(
    "../../data/processed/vietnam_dataset/validated_features.csv"
)

mapping_path = Path(
    "../../data/processed/feature_indicator_mapping.csv"
)

output_path = Path(
    "../../data/processed/vietnam_dataset/student_dbi_dimension_scores.csv"
)

print("Đã thiết lập đường dẫn dữ liệu.")

Đã thiết lập đường dẫn dữ liệu.


# 1. Tải dữ liệu

Phần này thực hiện tải toàn bộ dữ liệu cần thiết cho quá trình xây dựng chỉ số Digital Burnout.

Bao gồm:

- Bộ dữ liệu khảo sát đã làm sạch.
- Danh sách các đặc trưng đã được xác thực.
- Bảng ánh xạ giữa đặc trưng và Candidate Indicators.

Các dữ liệu này sẽ được sử dụng để tính toán điểm cho từng nhóm chỉ số DBI ở các phần tiếp theo.

In [5]:
# Đọc dữ liệu khảo sát

df = pd.read_csv(survey_path)

print("Kích thước dữ liệu:")

display(df.shape)

display(df.head())

Kích thước dữ liệu:


(590, 23)

,gender,birth_year,education_stage,work_mode,device_usage_type,daily_screen_time,social_media_hours,doomscrolling_duration,late_night_device_usage,notification_count,smartphone_unlocks,app_switch_frequency,concentration_score,distraction_frequency,focus_sessions,deep_work_hours,task_completion_rate,sleep_hours,sleep_quality,motivation_level,emotional_exhaustion,stress_level,mental_fatigue
0,Nam,2006 - 2008,Sinh viên đại học / cao đẳng (năm 3-4+),Hoàn toàn trực tiếp (đến trường / văn phòng ≥ ...,"Chủ yếu để giải trí (mạng xã hội, game, xem phim)",2,2,4,3,0,1,1,1,3,2,0,3,1,1,10,1,1,4
1,Nữ,2006 - 2008,Sinh viên đại học / cao đẳng (năm 1-2),"Kết hợp hybrid (2-3 ngày trực tiếp, còn lại on...",Cân bằng cả học lẫn giải trí,3,1,1,1,1,1,2,6,1,1,1,3,3,9,9,5,5,4
2,Nữ,2006 - 2008,Sinh viên đại học / cao đẳng (năm 3-4+),Hoàn toàn trực tiếp (đến trường / văn phòng ≥ ...,"Chủ yếu để học (LMS, tài liệu, nghiên cứu)",4,2,2,3,3,2,3,7,1,3,2,2,3,8,6,5,5,5
3,Nữ,2006 - 2008,Sinh viên đại học / cao đẳng (năm 1-2),"Tự do, không cố định lịch trình",Cân bằng cả học lẫn giải trí,3,2,1,3,1,1,1,6,1,3,1,3,1,4,8,4,5,3
4,Nữ,2009 - 2012,Học sinh THPT,Hoàn toàn trực tiếp (đến trường / văn phòng ≥ ...,"Chủ yếu để giải trí (mạng xã hội, game, xem phim)",4,3,4,3,2,1,1,7,1,2,1,0,3,8,4,2,2,4


In [6]:
# Đọc danh sách Validated Features

validated_features = pd.read_csv(
    validated_feature_path
)

display(validated_features)

,feature,selected
0,late_night_device_usage,True
1,stress_level,True
2,doomscrolling_duration,True
3,daily_screen_time,True
4,mental_fatigue,True
5,emotional_exhaustion,True
6,notification_count,True
7,app_switch_frequency,True
8,social_media_hours,True
9,concentration_score,True


In [7]:
# Đọc bảng Feature - Indicator Mapping

mapping_table = pd.read_csv(
    mapping_path
)

display(mapping_table.head())

,feature,indicator_id,indicator_name,dbi_dimension,mapping_type
0,daily_screen_time,DE01,Daily Total Screen Time,Digital Exposure,Direct
1,social_media_hours,DE02,Media Multitasking Frequency,Digital Exposure,Proxy
2,app_switch_frequency,DE02,Media Multitasking Frequency,Digital Exposure,Direct
3,notification_count,DE04,Notification Check Frequency,Digital Exposure,Direct
4,smartphone_unlocks,DE04,Notification Check Frequency,Digital Exposure,Proxy


In [8]:
# Danh sách Feature sử dụng

selected_features = validated_features[
    "feature"
].tolist()

print(f"Số lượng Feature: {len(selected_features)}")

selected_features

Số lượng Feature: 18


['late_night_device_usage',
 'stress_level',
 'doomscrolling_duration',
 'daily_screen_time',
 'mental_fatigue',
 'emotional_exhaustion',
 'notification_count',
 'app_switch_frequency',
 'social_media_hours',
 'concentration_score',
 'distraction_frequency',
 'focus_sessions',
 'sleep_quality',
 'task_completion_rate',
 'sleep_hours',
 'motivation_level',
 'smartphone_unlocks',
 'deep_work_hours']

# 2. Xây dựng chỉ số DBI

Phần này thực hiện chuẩn hóa các đặc trưng đã được xác thực và xây dựng điểm cho từng nhóm chỉ số Digital Burnout.

Để đảm bảo các đặc trưng có cùng thang đo, toàn bộ biến sẽ được chuẩn hóa về khoảng giá trị từ 0 đến 1 bằng phương pháp Min-Max Scaling.

Đối với các đặc trưng có tác động bảo vệ (giá trị càng cao thì nguy cơ Digital Burnout càng thấp), giá trị sẽ được đảo chiều trước khi tính điểm nhằm thống nhất hướng tác động của tất cả các chỉ số.

## 2.1 Chuẩn hóa các đặc trưng

Chuẩn hóa các đặc trưng đã được xác thực về cùng một thang đo từ 0 đến 1 bằng phương pháp Min-Max Scaling.

Việc chuẩn hóa giúp đảm bảo các chỉ số có đơn vị đo khác nhau vẫn có thể được tổng hợp để tính điểm DBI.

In [9]:
# Sao chép dữ liệu để chuẩn hóa

dbi_data = df.copy()

# Danh sách đặc trưng sử dụng

selected_features = validated_features["feature"].tolist()

# Khởi tạo Min-Max Scaler

scaler = MinMaxScaler()

# Chuẩn hóa dữ liệu

dbi_data[selected_features] = scaler.fit_transform(
    dbi_data[selected_features]
)

print("Đã chuẩn hóa các đặc trưng.")

display(
    dbi_data[selected_features].head()
)

Đã chuẩn hóa các đặc trưng.


,late_night_device_usage,stress_level,doomscrolling_duration,daily_screen_time,mental_fatigue,emotional_exhaustion,notification_count,app_switch_frequency,social_media_hours,concentration_score,distraction_frequency,focus_sessions,sleep_quality,task_completion_rate,sleep_hours,motivation_level,smartphone_unlocks,deep_work_hours
0,1.000000,0.00,1.00,0.50,0.75,0.00,0.000000,0.333333,0.666667,0.000000,1.000000,0.666667,0.000000,1.000000,0.25,1.000000,0.333333,0.000000
1,0.333333,1.00,0.25,0.75,0.75,1.00,0.333333,0.666667,0.333333,0.555556,0.333333,0.333333,0.888889,1.000000,0.75,0.888889,0.333333,0.333333
2,1.000000,1.00,0.50,1.00,1.00,1.00,1.000000,1.000000,0.666667,0.666667,0.333333,1.000000,0.777778,0.666667,0.75,0.555556,0.666667,0.666667
3,1.000000,1.00,0.25,0.75,0.50,0.75,0.333333,0.333333,0.666667,0.555556,0.333333,1.000000,0.333333,1.000000,0.25,0.777778,0.333333,0.333333
4,1.000000,0.25,1.00,1.00,0.75,0.25,0.666667,0.333333,1.000000,0.666667,0.333333,0.666667,0.777778,0.000000,0.75,0.333333,0.333333,0.333333


## 2.2 Đảo chiều các chỉ số bảo vệ

Một số đặc trưng phản ánh yếu tố bảo vệ đối với Digital Burnout.

Đối với các đặc trưng này, giá trị càng cao đồng nghĩa với nguy cơ Digital Burnout càng thấp.

Do đó, các chỉ số sẽ được đảo chiều để đảm bảo tất cả các đặc trưng đều có cùng hướng tác động, trong đó giá trị càng cao thể hiện nguy cơ Digital Burnout càng lớn.

In [10]:
# Khai báo các đặc trưng cần đảo chiều

protective_features = [

    "concentration_score",
    "focus_sessions",
    "deep_work_hours",
    "task_completion_rate",
    "sleep_hours",
    "sleep_quality"

]

print("Các đặc trưng cần đảo chiều:")

display(protective_features)

Các đặc trưng cần đảo chiều:


['concentration_score',
 'focus_sessions',
 'deep_work_hours',
 'task_completion_rate',
 'sleep_hours',
 'sleep_quality']

In [11]:
# Đảo chiều các đặc trưng bảo vệ

dbi_data[protective_features] = (
    1 - dbi_data[protective_features]
)

print("Đã đảo chiều các đặc trưng bảo vệ.")

display(
    dbi_data[
        protective_features
    ].head()
)

Đã đảo chiều các đặc trưng bảo vệ.


,concentration_score,focus_sessions,deep_work_hours,task_completion_rate,sleep_hours,sleep_quality
0,1.000000,0.333333,1.000000,0.000000,0.75,1.000000
1,0.444444,0.666667,0.666667,0.000000,0.25,0.111111
2,0.333333,0.000000,0.333333,0.333333,0.25,0.222222
3,0.444444,0.000000,0.666667,0.000000,0.75,0.666667
4,0.333333,0.333333,0.666667,1.000000,0.25,0.222222


# 3. Tính điểm các nhóm chỉ số DBI

Sau khi các đặc trưng đã được chuẩn hóa và thống nhất hướng tác động, phần này thực hiện tính điểm cho từng nhóm chỉ số của Digital Burnout Indicator (DBI).

Các đặc trưng thuộc cùng một nhóm chỉ số sẽ được tổng hợp bằng phương pháp trung bình cộng (Mean Aggregation). Ở giai đoạn này, tất cả các đặc trưng trong cùng một nhóm được gán trọng số bằng nhau.

Việc xác định trọng số chính thức cho từng nhóm chỉ số sẽ được thực hiện trong notebook xây dựng DBI Framework.

## 3.1 Xây dựng nhóm đặc trưng theo DBI Dimension

Các đặc trưng đã được xác thực sẽ được phân nhóm theo bốn thành phần của Digital Burnout Indicator (DBI), bao gồm:

- Digital Exposure
- Cognitive Performance
- Psychological Symptoms
- Sleep & Recovery

Việc phân nhóm được thực hiện dựa trên bảng Feature–Indicator Mapping đã xây dựng ở Notebook Feature Validation.

In [12]:
# Xây dựng danh sách đặc trưng theo từng nhóm DBI

dbi_groups = {}

for dimension in mapping_table["dbi_dimension"].unique():

    dbi_groups[dimension] = (
        mapping_table[
            mapping_table["dbi_dimension"] == dimension
        ]["feature"]
        .tolist()
    )

print("Các nhóm chỉ số DBI:")

for dimension, features in dbi_groups.items():
    print(f"\n{dimension}")
    for feature in features:
        print(f"  - {feature}")

Các nhóm chỉ số DBI:

Digital Exposure
  - daily_screen_time
  - social_media_hours
  - app_switch_frequency
  - notification_count
  - smartphone_unlocks
  - late_night_device_usage

Psychological Symptoms
  - doomscrolling_duration
  - stress_level
  - emotional_exhaustion
  - mental_fatigue

Cognitive Performance
  - distraction_frequency
  - concentration_score
  - focus_sessions
  - deep_work_hours
  - task_completion_rate

Sleep & Recovery
  - sleep_hours
  - sleep_quality


## 3.2 Tính điểm từng nhóm DBI

Điểm của mỗi nhóm chỉ số được tính bằng giá trị trung bình của các đặc trưng thuộc nhóm đó sau khi đã chuẩn hóa và thống nhất hướng tác động.

Ở giai đoạn này, tất cả các đặc trưng trong cùng một nhóm được xem có mức độ đóng góp như nhau.

In [13]:
# Tính điểm cho từng nhóm DBI

for dimension, features in dbi_groups.items():

    score_name = (
        dimension
        .lower()
        .replace(" ", "_")
        .replace("&", "")
        .replace("__", "_")
        + "_score"
    )

    dbi_data[score_name] = (
        dbi_data[features]
        .mean(axis=1)
    )

print("Đã tính điểm các nhóm DBI.")

Đã tính điểm các nhóm DBI.


## 3.3 Tổng hợp kết quả

Tổng hợp điểm của bốn nhóm chỉ số Digital Burnout để phục vụ cho quá trình xây dựng DBI Framework ở các giai đoạn tiếp theo.

In [14]:
# Danh sách các cột điểm DBI

dimension_scores = [
    col
    for col in dbi_data.columns
    if col.endswith("_score")
]

display(
    dbi_data[
        dimension_scores
    ].head()
)

,concentration_score,digital_exposure_score,psychological_symptoms_score,cognitive_performance_score,sleep_recovery_score
0,1.000000,0.472222,0.4375,0.666667,0.875000
1,0.444444,0.458333,0.7500,0.422222,0.180556
2,0.333333,0.888889,0.8750,0.266667,0.236111
3,0.444444,0.569444,0.6250,0.288889,0.708333
4,0.333333,0.722222,0.5625,0.533333,0.236111


# 4. Tổng hợp kết quả đánh giá DBI

Phần này tổng hợp và kiểm tra kết quả sau khi hoàn thành quá trình chuẩn hóa các đặc trưng và tính điểm cho từng nhóm chỉ số Digital Burnout.

Các thống kê mô tả và bộ dữ liệu đánh giá DBI được trình bày nhằm đánh giá tính hợp lý của các điểm số trước khi sử dụng trong giai đoạn xây dựng **Digital Burnout Indicator (DBI) Framework** và phân tích so sánh giữa bộ dữ liệu quốc tế và dữ liệu khảo sát sinh viên Việt Nam.

## 4.1 Phân bố điểm các nhóm chỉ số DBI

Phần này trình bày thống kê mô tả của bốn nhóm chỉ số Digital Burnout nhằm đánh giá phân bố điểm trong bộ dữ liệu khảo sát.

Các kết quả này giúp kiểm tra tính hợp lý của quá trình chuẩn hóa và tính điểm trước khi xây dựng DBI Framework.

In [15]:
# Danh sách các cột điểm DBI

dimension_scores = [

    "digital_exposure_score",
    "cognitive_performance_score",
    "psychological_symptoms_score",
    "sleep_recovery_score"

]

display(
    dbi_data[dimension_scores]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
digital_exposure_score,590.0,0.605085,0.120240,0.041667,0.555556,0.597222,0.666667,1.0
cognitive_performance_score,590.0,0.496798,0.133432,0.066667,0.422222,0.488889,0.577778,1.0
psychological_symptoms_score,590.0,0.680191,0.161626,0.062500,0.562500,0.687500,0.812500,1.0
sleep_recovery_score,590.0,0.558569,0.182128,0.000000,0.430556,0.583333,0.708333,1.0


## 4.2 Bộ dữ liệu đánh giá DBI

Tổng hợp bộ dữ liệu đánh giá DBI bao gồm thông tin khảo sát ban đầu và điểm của bốn nhóm chỉ số Digital Burnout.

Bộ dữ liệu này sẽ được sử dụng trong giai đoạn xây dựng DBI Framework và phân tích so sánh giữa bộ dữ liệu quốc tế và khảo sát sinh viên Việt Nam.

In [16]:
display(
    dbi_data.head()
)

,gender,birth_year,education_stage,work_mode,device_usage_type,daily_screen_time,social_media_hours,doomscrolling_duration,late_night_device_usage,notification_count,smartphone_unlocks,app_switch_frequency,concentration_score,distraction_frequency,focus_sessions,deep_work_hours,task_completion_rate,sleep_hours,sleep_quality,motivation_level,emotional_exhaustion,stress_level,mental_fatigue,digital_exposure_score,psychological_symptoms_score,cognitive_performance_score,sleep_recovery_score
0,Nam,2006 - 2008,Sinh viên đại học / cao đẳng (năm 3-4+),Hoàn toàn trực tiếp (đến trường / văn phòng ≥ ...,"Chủ yếu để giải trí (mạng xã hội, game, xem phim)",0.50,0.666667,1.00,1.000000,0.000000,0.333333,0.333333,1.000000,1.000000,0.333333,1.000000,0.000000,0.75,1.000000,1.000000,0.00,0.00,0.75,0.472222,0.4375,0.666667,0.875000
1,Nữ,2006 - 2008,Sinh viên đại học / cao đẳng (năm 1-2),"Kết hợp hybrid (2-3 ngày trực tiếp, còn lại on...",Cân bằng cả học lẫn giải trí,0.75,0.333333,0.25,0.333333,0.333333,0.333333,0.666667,0.444444,0.333333,0.666667,0.666667,0.000000,0.25,0.111111,0.888889,1.00,1.00,0.75,0.458333,0.7500,0.422222,0.180556
2,Nữ,2006 - 2008,Sinh viên đại học / cao đẳng (năm 3-4+),Hoàn toàn trực tiếp (đến trường / văn phòng ≥ ...,"Chủ yếu để học (LMS, tài liệu, nghiên cứu)",1.00,0.666667,0.50,1.000000,1.000000,0.666667,1.000000,0.333333,0.333333,0.000000,0.333333,0.333333,0.25,0.222222,0.555556,1.00,1.00,1.00,0.888889,0.8750,0.266667,0.236111
3,Nữ,2006 - 2008,Sinh viên đại học / cao đẳng (năm 1-2),"Tự do, không cố định lịch trình",Cân bằng cả học lẫn giải trí,0.75,0.666667,0.25,1.000000,0.333333,0.333333,0.333333,0.444444,0.333333,0.000000,0.666667,0.000000,0.75,0.666667,0.777778,0.75,1.00,0.50,0.569444,0.6250,0.288889,0.708333
4,Nữ,2009 - 2012,Học sinh THPT,Hoàn toàn trực tiếp (đến trường / văn phòng ≥ ...,"Chủ yếu để giải trí (mạng xã hội, game, xem phim)",1.00,1.000000,1.00,1.000000,0.666667,0.333333,0.333333,0.333333,0.333333,0.333333,0.666667,1.000000,0.25,0.222222,0.333333,0.25,0.25,0.75,0.722222,0.5625,0.533333,0.236111


# 5. Xuất bộ dữ liệu đánh giá DBI

Lưu bộ dữ liệu đánh giá DBI sau khi hoàn thành quá trình chuẩn hóa và tính điểm các nhóm chỉ số.

Bộ dữ liệu này sẽ là đầu vào cho notebook xây dựng DBI Framework.

In [17]:
output_path = Path(
    "../../data/processed/vietnam_dataset/student_dbi_dimension_scores.csv"
)

dbi_data.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Đã lưu kết quả tại: {output_path}")

Đã lưu kết quả tại: ..\..\data\processed\vietnam_dataset\student_dbi_dimension_scores.csv
